# Data Quality & Expectations

## Objetivo
Evaluar controles de calidad sobre las tablas Bronze y Silver del pipeline
de Applebee's antes de la generación de Gold.

## Base de implementación
Este notebook sigue la implementación de los notebooks 06 y 07:
- Catálogo y esquema: workspace.default.
- Prefijo de tablas: p2_team_.
- Entradas: cinco tablas Bronze y cinco tablas Silver.
- Parámetro de ejecución: stage=quality_gate.
- No crea, modifica ni elimina registros de las tablas evaluadas.

## Controles implementados
1. Accesibilidad de las diez tablas.
2. Identificadores obligatorios y unicidad de Silver checks.
3. Reconciliación de conteos y llaves entre Bronze y Silver checks.
4. Integridad de eventos respecto a su check padre.
5. Reconciliación global de conteos de eventos.
6. Unicidad de eventos con posición: artículos, pagos y voids.
7. Consistencia de siete importes entre detalle y resumen Silver por check.
8. Validez y consistencia temporal.
9. Consistencia de seis banderas de anomalías.

## Criterios de ejecución
- Una regla crítica incumplida provoca una excepción.
- Las advertencias temporales se reportan sin bloquear.
- Si todos los controles implementados pasan, el notebook termina
  normalmente e imprime: QUALITY GATE: PASSED.
- Las validaciones son controles programados en un notebook;
  no constituyen expectations nativas de Lakeflow.

## Resultado de la ejecución manual completa
- 80 reglas evaluadas.
- 0 reglas fallidas.
- 1 regla con advertencia.
- 52,503 de 2,142,846 checks tienen apertura igual a cierre: aproximadamente 2.45%.
- Estos registros se conservan y no representan fraude confirmado.

## Limitaciones
- No se valida unicidad por posición de descuentos, porque el esquema
  actual no conserva esa posición.
- Los conteos de eventos Bronze/Silver se comparan globalmente,
  no por check.
- La reconciliación financiera comprueba consistencia interna de Silver,
  no fidelidad de importes contra Bronze.
- Las relaciones con dimensiones están pendientes de evaluar.
- Se generó un inventario de tipos, pero no se implementó un contrato
  automático de tipos.
- No se verificó la zona horaria del origen.

## Estado de integración
- Ejecución manual completa: satisfactoria.
- Ejecución desde Databricks Jobs: pendiente.
- Prueba de bloqueo de Gold ante fallo del quality gate: pendiente.

In [0]:
# 08. Data Quality & Expectations
# Configuración para revisar la implementación actual del notebook 07.

dbutils.widgets.text("stage", "quality_gate", "Etapa de ejecución")
stage = dbutils.widgets.get("stage")

if stage != "quality_gate":
    raise ValueError(
        f"Etapa no válida: {stage!r}. Este notebook solo acepta stage=quality_gate."
    )

CATALOG = "workspace"
SCHEMA = "default"
DATA_PREFIX = "p2_team_"


def full_table_name(name):
    return f"{CATALOG}.{SCHEMA}.{DATA_PREFIX}{name}"


test_table = full_table_name("silver_checks")

print(f"Etapa: {stage}")
print(f"Tabla de prueba: {test_table}")

try:
    spark.table(test_table).limit(1).collect()
except Exception as error:
    detail = str(error).split("JVM stacktrace:")[0].strip()
    raise RuntimeError(f"No se pudo leer {test_table}.\n{detail}") from None

print("ACCESO CONFIRMADO: Silver checks se puede consultar.")
print("Pendientes las validaciones de contenido conforme a los notebooks 06 y 07.")

Etapa: quality_gate
Tabla de prueba: workspace.default.p2_team_silver_checks
ACCESO CONFIRMADO: Silver checks se puede consultar.
Pendientes las validaciones de contenido conforme a los notebooks 06 y 07.


In [0]:
# Inventario de la implementación actual del notebook 07.
# Solo lectura: no crea ni modifica tablas.

BRONZE_TABLES = [
    "bronze_checks",
    "bronze_employees",
    "bronze_items",
    "bronze_locations",
    "bronze_revenue_centers",
]

SILVER_TABLES = [
    "silver_checks",
    "silver_items_sold",
    "silver_discounts",
    "silver_payments",
    "silver_voids",
]

table_access_results = []

for name in BRONZE_TABLES + SILVER_TABLES:
    table = full_table_name(name)

    try:
        spark.table(table).limit(1).collect()
        table_access_results.append(
            (table, "PASS", "Lectura disponible")
        )

    except Exception as error:
        detail = str(error).split("JVM stacktrace:")[0].strip()
        table_access_results.append(
            (table, "FAIL", detail[:2000])
        )

access_report = spark.createDataFrame(
    table_access_results,
    schema="table_name STRING, status STRING, detail STRING",
)

display(access_report)

failed_tables = [
    table
    for table, status, detail in table_access_results
    if status == "FAIL"
]

if failed_tables:
    raise RuntimeError(
        "QUALITY GATE: FAILED — "
        f"No se pudieron leer {len(failed_tables)} "
        f"de {len(table_access_results)} tablas requeridas "
        "por la implementación actual. Revisa el reporte."
    )

print("ACCESO CONFIRMADO: las 10 tablas se pueden consultar.")
print("Pendientes las validaciones de contenido y conformidad con el acuerdo.")

table_name,status,detail
workspace.default.p2_team_bronze_checks,PASS,Lectura disponible
workspace.default.p2_team_bronze_employees,PASS,Lectura disponible
workspace.default.p2_team_bronze_items,PASS,Lectura disponible
workspace.default.p2_team_bronze_locations,PASS,Lectura disponible
workspace.default.p2_team_bronze_revenue_centers,PASS,Lectura disponible
workspace.default.p2_team_silver_checks,PASS,Lectura disponible
workspace.default.p2_team_silver_items_sold,PASS,Lectura disponible
workspace.default.p2_team_silver_discounts,PASS,Lectura disponible
workspace.default.p2_team_silver_payments,PASS,Lectura disponible
workspace.default.p2_team_silver_voids,PASS,Lectura disponible


ACCESO CONFIRMADO: las 10 tablas se pueden consultar.
Pendientes las validaciones de contenido y conformidad con el acuerdo.


In [0]:
# Inventario de columnas y tipos.
# No modifica tablas ni muestra datos personales.

schema_rows = []

for name in BRONZE_TABLES + SILVER_TABLES:
    table = full_table_name(name)
    df = spark.table(table)

    for position, field in enumerate(df.schema.fields, start=1):
        schema_rows.append(
            (
                table,
                position,
                field.name,
                field.dataType.simpleString(),
                field.nullable,
            )
        )

schema_report = spark.createDataFrame(
    schema_rows,
    schema=(
        "table_name STRING, column_position INT, "
        "column_name STRING, data_type STRING, nullable BOOLEAN"
    ),
)

display(
    schema_report.orderBy("table_name", "column_position")
)

print("INVENTARIO DE ESQUEMAS COMPLETADO.")
print("Todavía no se han medido nulos, duplicados ni integridad.")

table_name,column_position,column_name,data_type,nullable
workspace.default.p2_team_bronze_checks,1,BusinessDate,string,true
workspace.default.p2_team_bronze_checks,2,CloseTime,string,true
workspace.default.p2_team_bronze_checks,3,Discounts,array>,true
workspace.default.p2_team_bronze_checks,4,Id,bigint,true
workspace.default.p2_team_bronze_checks,5,ItemsSold,"array>,EmployeeId:bigint,GrossPrice:double,ItemId:bigint,ItemMajorCatId:bigint,ItemMajorCatName:string,ItemName:string,ItemPLU:bigint,ItemSubCatId:bigint,ItemSubCatName:string,OrderTime:string,ParentSequence:bigint,PosSourceId:bigint,Quantity:double,RevenueCenterId:bigint,Seat:bigint,Sequence:bigint,SoldPrice:double>>",true
workspace.default.p2_team_bronze_checks,6,LocationId,bigint,true
workspace.default.p2_team_bronze_checks,7,LoyaltyId,string,true
workspace.default.p2_team_bronze_checks,8,OpenTime,string,true
workspace.default.p2_team_bronze_checks,9,Payments,array>,true
workspace.default.p2_team_bronze_checks,10,RevenueCenterId,bigint,true


INVENTARIO DE ESQUEMAS COMPLETADO.
Todavía no se han medido nulos, duplicados ni integridad.


In [0]:
from pyspark.sql import functions as F

checks = spark.table(full_table_name("silver_checks"))

required_identifiers = [
    "check_key",
    "LocationId",
    "check_id",
    "business_date",
    "BusinessDate",
]

# Total y nulos por identificador.
key_summary = checks.agg(
    F.count("*").alias("total_rows"),
    *[
        F.count(
            F.when(F.col(column).isNull(), 1)
        ).alias(f"null_{column}")
        for column in required_identifiers
    ],
    F.count(
        F.when(F.trim(F.col("check_key")) == "", 1)
    ).alias("blank_check_key"),
).first().asDict()

# Grupos de llaves que aparecen más de una vez.
duplicate_check_key_groups = (
    checks.groupBy("check_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_composite_groups = (
    checks.groupBy("LocationId", "BusinessDate", "check_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Reporte de este bloque.
key_rule_results = []

def add_key_rule(rule, observed, expected=0):
    key_rule_results.append(
        (
            rule,
            int(observed),
            int(expected),
            "PASS" if observed == expected else "FAIL",
        )
    )

add_key_rule(
    "La tabla contiene registros",
    1 if key_summary["total_rows"] > 0 else 0,
    expected=1,
)

for column in required_identifiers:
    add_key_rule(
        f"Identificador sin nulos: {column}",
        key_summary[f"null_{column}"],
    )

add_key_rule(
    "check_key sin cadenas vacías",
    key_summary["blank_check_key"],
)

add_key_rule(
    "Grupos duplicados por check_key",
    duplicate_check_key_groups,
)

add_key_rule(
    "Grupos duplicados por LocationId + BusinessDate + check_id",
    duplicate_composite_groups,
)

key_report = spark.createDataFrame(
    key_rule_results,
    schema=(
        "rule STRING, observed_value LONG, "
        "expected_value LONG, status STRING"
    ),
)

print(f"Tabla: {full_table_name('silver_checks')}")
print(f"Filas evaluadas: {key_summary['total_rows']:,}")
display(key_report)

failed_key_rules = [
    rule
    for rule, observed, expected, status in key_rule_results
    if status == "FAIL"
]

if failed_key_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Reglas de llaves incumplidas: "
        + "; ".join(failed_key_rules)
    )

print("BLOQUE DE LLAVES: PASSED.")
print("Pendientes reconciliación, relaciones y demás controles.")

Tabla: workspace.default.p2_team_silver_checks
Filas evaluadas: 2,142,846


rule,observed_value,expected_value,status
La tabla contiene registros,1,1,PASS
Identificador sin nulos: check_key,0,0,PASS
Identificador sin nulos: LocationId,0,0,PASS
Identificador sin nulos: check_id,0,0,PASS
Identificador sin nulos: business_date,0,0,PASS
Identificador sin nulos: BusinessDate,0,0,PASS
check_key sin cadenas vacías,0,0,PASS
Grupos duplicados por check_key,0,0,PASS
Grupos duplicados por LocationId + BusinessDate + check_id,0,0,PASS


BLOQUE DE LLAVES: PASSED.
Pendientes reconciliación, relaciones y demás controles.


In [0]:
# Reconciliación de checks entre Bronze y Silver.
# No modifica datos.

bronze_checks = spark.table(full_table_name("bronze_checks"))
silver_checks = spark.table(full_table_name("silver_checks"))

bronze_count = bronze_checks.count()
silver_count = silver_checks.count()

# Alinear nombres sin concatenar ni convertir los identificadores.
bronze_keys = bronze_checks.select(
    "LocationId",
    "BusinessDate",
    F.col("Id").alias("check_id"),
).distinct()

silver_keys = silver_checks.select(
    "LocationId",
    "BusinessDate",
    "check_id",
).distinct()

join_columns = ["LocationId", "BusinessDate", "check_id"]

missing_in_silver = (
    bronze_keys
    .join(silver_keys, on=join_columns, how="left_anti")
    .count()
)

extra_in_silver = (
    silver_keys
    .join(bronze_keys, on=join_columns, how="left_anti")
    .count()
)

reconciliation_results = [
    (
        "Diferencia absoluta de filas Bronze vs Silver",
        abs(bronze_count - silver_count),
        0,
    ),
    (
        "Llaves de Bronze ausentes en Silver",
        missing_in_silver,
        0,
    ),
    (
        "Llaves de Silver ausentes en Bronze",
        extra_in_silver,
        0,
    ),
]

reconciliation_rows = [
    (
        rule,
        int(observed),
        int(expected),
        "PASS" if observed == expected else "FAIL",
    )
    for rule, observed, expected in reconciliation_results
]

reconciliation_report = spark.createDataFrame(
    reconciliation_rows,
    schema=(
        "rule STRING, observed_value LONG, "
        "expected_value LONG, status STRING"
    ),
)

print(f"Checks Bronze: {bronze_count:,}")
print(f"Checks Silver: {silver_count:,}")
display(reconciliation_report)

failed_reconciliation_rules = [
    rule
    for rule, observed, expected, status in reconciliation_rows
    if status == "FAIL"
]

if failed_reconciliation_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Reconciliación incumplida: "
        + "; ".join(failed_reconciliation_rules)
    )

print("BLOQUE DE RECONCILIACIÓN CHECKS: PASSED.")
print("Pendientes relaciones de eventos y demás controles.")

Checks Bronze: 2,142,846
Checks Silver: 2,142,846


rule,observed_value,expected_value,status
Diferencia absoluta de filas Bronze vs Silver,0,0,PASS
Llaves de Bronze ausentes en Silver,0,0,PASS
Llaves de Silver ausentes en Bronze,0,0,PASS


BLOQUE DE RECONCILIACIÓN CHECKS: PASSED.
Pendientes relaciones de eventos y demás controles.


In [0]:
# Integridad de eventos contra Silver checks.
# Solo lectura; no elimina ni modifica registros.

from functools import reduce

EVENT_TABLES = [
    "silver_items_sold",
    "silver_discounts",
    "silver_payments",
    "silver_voids",
]

event_identifiers = [
    "check_key",
    "LocationId",
    "check_id",
    "business_date",
]

parent_checks = spark.table(
    full_table_name("silver_checks")
).select(
    "check_key",
    F.col("LocationId").alias("parent_location_id"),
    F.col("check_id").alias("parent_check_id"),
    F.col("business_date").alias("parent_business_date"),
    F.lit(1).alias("parent_exists"),
)

event_integrity_rows = []

for name in EVENT_TABLES:
    events = spark.table(full_table_name(name))

    joined = events.select(*event_identifiers).join(
        parent_checks,
        on="check_key",
        how="left",
    )

    null_identifier = reduce(
        lambda a, b: a | b,
        [F.col(c).isNull() for c in event_identifiers],
    )

    blank_key = F.trim(F.col("check_key")) == ""
    parent_found = F.col("parent_exists").isNotNull()

    parent_mismatch = parent_found & (
        ~F.col("LocationId").eqNullSafe(F.col("parent_location_id"))
        | ~F.col("check_id").eqNullSafe(F.col("parent_check_id"))
        | ~F.col("business_date").eqNullSafe(
            F.col("parent_business_date")
        )
    )

    summary = joined.agg(
        F.count("*").alias("total"),
        F.count(F.when(null_identifier, 1)).alias("null_identifiers"),
        F.count(F.when(blank_key, 1)).alias("blank_keys"),
        F.count(
            F.when(~parent_found, 1)
        ).alias("missing_parent"),
        F.count(
            F.when(parent_mismatch, 1)
        ).alias("parent_mismatch"),
    ).first().asDict()

    rules = [
        ("Eventos con identificadores nulos", summary["null_identifiers"]),
        ("Eventos con check_key vacío", summary["blank_keys"]),
        ("Eventos sin check padre", summary["missing_parent"]),
        ("Eventos con identificadores distintos al padre",
         summary["parent_mismatch"]),
    ]

    for rule, invalid_rows in rules:
        event_integrity_rows.append(
            (
                full_table_name(name),
                rule,
                int(summary["total"]),
                int(invalid_rows),
                "PASS" if invalid_rows == 0 else "FAIL",
            )
        )

event_integrity_report = spark.createDataFrame(
    event_integrity_rows,
    schema=(
        "table_name STRING, rule STRING, "
        "evaluated_rows LONG, invalid_rows LONG, status STRING"
    ),
)

display(event_integrity_report)

failed_event_rules = [
    f"{table}: {rule}"
    for table, rule, total, invalid, status in event_integrity_rows
    if status == "FAIL"
]

if failed_event_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Integridad de eventos incumplida: "
        + "; ".join(failed_event_rules)
    )

print("BLOQUE DE INTEGRIDAD EVENTOS → CHECKS: PASSED.")
print("Pendientes reconciliación de eventos y demás controles.")

table_name,rule,evaluated_rows,invalid_rows,status
workspace.default.p2_team_silver_items_sold,Eventos con identificadores nulos,28558925,0,PASS
workspace.default.p2_team_silver_items_sold,Eventos con check_key vacío,28558925,0,PASS
workspace.default.p2_team_silver_items_sold,Eventos sin check padre,28558925,0,PASS
workspace.default.p2_team_silver_items_sold,Eventos con identificadores distintos al padre,28558925,0,PASS
workspace.default.p2_team_silver_discounts,Eventos con identificadores nulos,516690,0,PASS
workspace.default.p2_team_silver_discounts,Eventos con check_key vacío,516690,0,PASS
workspace.default.p2_team_silver_discounts,Eventos sin check padre,516690,0,PASS
workspace.default.p2_team_silver_discounts,Eventos con identificadores distintos al padre,516690,0,PASS
workspace.default.p2_team_silver_payments,Eventos con identificadores nulos,2232092,0,PASS
workspace.default.p2_team_silver_payments,Eventos con check_key vacío,2232092,0,PASS


BLOQUE DE INTEGRIDAD EVENTOS → CHECKS: PASSED.
Pendientes reconciliación de eventos y demás controles.


In [0]:
# Reconciliación de conteos de eventos Bronze → Silver.
# Solo lectura; no guarda ni modifica tablas.

bronze = spark.table(full_table_name("bronze_checks"))

expected_items = bronze.select(
    F.explode("ItemsSold").alias("item")
).count()

expected_payments = bronze.select(
    F.explode("Payments").alias("payment")
).count()

expected_voids = bronze.select(
    F.explode("Voids").alias("void")
).count()

expected_check_discounts = bronze.select(
    F.explode("Discounts").alias("discount")
).count()

expected_item_discounts = (
    bronze
    .select(F.explode("ItemsSold").alias("item"))
    .select(F.explode("item.Discounts").alias("discount"))
    .count()
)

discounts = spark.table(full_table_name("silver_discounts"))

discount_summary = discounts.agg(
    F.count("*").alias("total"),
    F.count(
        F.when(F.col("discount_level") == "check", 1)
    ).alias("check"),
    F.count(
        F.when(F.col("discount_level") == "item", 1)
    ).alias("item"),
    F.count(
        F.when(
            F.col("discount_level").isNull()
            | ~F.col("discount_level").isin("check", "item"),
            1,
        )
    ).alias("invalid_level"),
).first().asDict()

event_count_pairs = [
    (
        "Artículos vendidos",
        expected_items,
        spark.table(full_table_name("silver_items_sold")).count(),
    ),
    (
        "Pagos",
        expected_payments,
        spark.table(full_table_name("silver_payments")).count(),
    ),
    (
        "Voids",
        expected_voids,
        spark.table(full_table_name("silver_voids")).count(),
    ),
    (
        "Descuentos de cuenta",
        expected_check_discounts,
        discount_summary["check"],
    ),
    (
        "Descuentos de artículo",
        expected_item_discounts,
        discount_summary["item"],
    ),
    (
        "Descuentos totales",
        expected_check_discounts + expected_item_discounts,
        discount_summary["total"],
    ),
    (
        "Descuentos con nivel inválido o nulo",
        0,
        discount_summary["invalid_level"],
    ),
]

event_count_rows = [
    (
        rule,
        int(expected),
        int(actual),
        int(actual - expected),
        "PASS" if actual == expected else "FAIL",
    )
    for rule, expected, actual in event_count_pairs
]

event_count_report = spark.createDataFrame(
    event_count_rows,
    schema=(
        "rule STRING, expected_rows LONG, actual_rows LONG, "
        "difference LONG, status STRING"
    ),
)

display(event_count_report)

failed_event_count_rules = [
    rule
    for rule, expected, actual, difference, status in event_count_rows
    if status == "FAIL"
]

if failed_event_count_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Conteos de eventos incumplidos: "
        + "; ".join(failed_event_count_rules)
    )

print("BLOQUE DE CONTEOS DE EVENTOS: PASSED.")
print("Pendientes duplicados de eventos, importes y demás controles.")

rule,expected_rows,actual_rows,difference,status
Artículos vendidos,28558925,28558925,0,PASS
Pagos,2232092,2232092,0,PASS
Voids,178303,178303,0,PASS
Descuentos de cuenta,240699,240699,0,PASS
Descuentos de artículo,275991,275991,0,PASS
Descuentos totales,516690,516690,0,PASS
Descuentos con nivel inválido o nulo,0,0,0,PASS


BLOQUE DE CONTEOS DE EVENTOS: PASSED.
Pendientes duplicados de eventos, importes y demás controles.


In [0]:
# Unicidad y validez básica de posiciones de eventos.
# Solo lectura.

POSITION_COLUMNS = {
    "silver_items_sold": "item_pos",
    "silver_payments": "payment_pos",
    "silver_voids": "void_pos",
}

event_position_rows = []

for name, position_column in POSITION_COLUMNS.items():
    events = spark.table(full_table_name(name))

    position_summary = events.agg(
        F.count("*").alias("total"),
        F.count(
            F.when(F.col(position_column).isNull(), 1)
        ).alias("null_positions"),
        F.count(
            F.when(F.col(position_column) < 0, 1)
        ).alias("negative_positions"),
    ).first().asDict()

    duplicate_summary = (
        events
        .groupBy("check_key", position_column)
        .count()
        .filter(F.col("count") > 1)
        .agg(
            F.count("*").alias("duplicate_groups"),
            F.coalesce(
                F.sum(F.col("count") - 1),
                F.lit(0),
            ).alias("excess_rows"),
        )
        .first()
        .asDict()
    )

    rules = [
        ("Posiciones nulas", position_summary["null_positions"]),
        ("Posiciones negativas", position_summary["negative_positions"]),
        ("Grupos de llave de evento duplicados",
         duplicate_summary["duplicate_groups"]),
        ("Filas sobrantes por llave duplicada",
         duplicate_summary["excess_rows"]),
    ]

    for rule, observed in rules:
        event_position_rows.append(
            (
                full_table_name(name),
                rule,
                int(position_summary["total"]),
                int(observed),
                "PASS" if observed == 0 else "FAIL",
            )
        )

event_position_report = spark.createDataFrame(
    event_position_rows,
    schema=(
        "table_name STRING, rule STRING, "
        "evaluated_rows LONG, observed_value LONG, status STRING"
    ),
)

display(event_position_report)

failed_position_rules = [
    f"{table}: {rule}"
    for table, rule, total, observed, status in event_position_rows
    if status == "FAIL"
]

if failed_position_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Llaves de eventos incumplidas: "
        + "; ".join(failed_position_rules)
    )

print("BLOQUE DE LLAVES DE EVENTOS CON POSICIÓN: PASSED.")
print("Descuentos: unicidad por posición no evaluable con el esquema actual.")
print("Pendientes importes y demás controles.")

table_name,rule,evaluated_rows,observed_value,status
workspace.default.p2_team_silver_items_sold,Posiciones nulas,28558925,0,PASS
workspace.default.p2_team_silver_items_sold,Posiciones negativas,28558925,0,PASS
workspace.default.p2_team_silver_items_sold,Grupos de llave de evento duplicados,28558925,0,PASS
workspace.default.p2_team_silver_items_sold,Filas sobrantes por llave duplicada,28558925,0,PASS
workspace.default.p2_team_silver_payments,Posiciones nulas,2232092,0,PASS
workspace.default.p2_team_silver_payments,Posiciones negativas,2232092,0,PASS
workspace.default.p2_team_silver_payments,Grupos de llave de evento duplicados,2232092,0,PASS
workspace.default.p2_team_silver_payments,Filas sobrantes por llave duplicada,2232092,0,PASS
workspace.default.p2_team_silver_voids,Posiciones nulas,178303,0,PASS
workspace.default.p2_team_silver_voids,Posiciones negativas,178303,0,PASS


BLOQUE DE LLAVES DE EVENTOS CON POSICIÓN: PASSED.
Descuentos: unicidad por posición no evaluable con el esquema actual.
Pendientes importes y demás controles.


In [0]:
# Reconciliación financiera por check dentro de Silver.
# Tolerancia propuesta: 0.01 unidades monetarias.
# No modifica datos.

AMOUNT_TOLERANCE = 0.01

amount_checks = [
    ("Venta bruta", "silver_items_sold",
     "GrossPrice", "total_gross_price", None),

    ("Venta registrada", "silver_items_sold",
     "SoldPrice", "total_sold_price", None),

    ("Pagos", "silver_payments",
     "Amount", "payment_amount", None),

    ("Propinas registradas", "silver_payments",
     "Gratuity", "gratuity_raw", None),

    ("Voids", "silver_voids",
     "Amount", "void_amount", None),

    ("Descuentos de cuenta", "silver_discounts",
     "Amount", "check_discount_amount", "check"),

    ("Descuentos de artículo", "silver_discounts",
     "Amount", "item_discount_amount", "item"),
]

amount_rows = []

for label, table_name, detail_column, header_column, level in amount_checks:
    detail = spark.table(full_table_name(table_name))

    if level is not None:
        detail = detail.filter(F.col("discount_level") == level)

    detail_totals = detail.groupBy("check_key").agg(
        F.sum(
            F.coalesce(F.col(detail_column), F.lit(0.0))
        ).alias("detail_total")
    )

    comparison = (
        spark.table(full_table_name("silver_checks"))
        .select(
            "check_key",
            F.col(header_column).alias("header_total"),
        )
        .join(detail_totals, on="check_key", how="left")
        .withColumn(
            "detail_total",
            F.coalesce(F.col("detail_total"), F.lit(0.0)),
        )
    )

    header = F.col("header_total")
    detail_total = F.col("detail_total")

    invalid_amount = (
        header.isNull()
        | F.isnan(header)
        | F.isnan(detail_total)
        | (F.abs(header) == F.lit(float("inf")))
        | (F.abs(detail_total) == F.lit(float("inf")))
    )

    difference = F.abs(header - detail_total)

    summary = comparison.agg(
        F.count("*").alias("evaluated"),
        F.count(
            F.when(invalid_amount, 1)
        ).alias("invalid_amounts"),
        F.count(
            F.when(
                (~invalid_amount) & (difference > AMOUNT_TOLERANCE),
                1,
            )
        ).alias("different_checks"),
        F.max(
            F.when(~invalid_amount, difference)
        ).alias("max_difference"),
    ).first().asDict()

    passed = (
        summary["invalid_amounts"] == 0
        and summary["different_checks"] == 0
    )

    amount_rows.append(
        (
            label,
            int(summary["evaluated"]),
            int(summary["invalid_amounts"]),
            int(summary["different_checks"]),
            (
                float(summary["max_difference"])
                if summary["max_difference"] is not None
                else None
            ),
            "PASS" if passed else "FAIL",
        )
    )

amount_report = spark.createDataFrame(
    amount_rows,
    schema=(
        "metric STRING, evaluated_checks LONG, "
        "invalid_aggregate_amounts LONG, different_checks LONG, "
        "max_absolute_difference DOUBLE, status STRING"
    ),
)

display(amount_report)

failed_amount_rules = [
    row[0] for row in amount_rows if row[-1] == "FAIL"
]

if failed_amount_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Importes inconsistentes: "
        + "; ".join(failed_amount_rules)
    )

print("BLOQUE DE IMPORTES DETALLE → CHECKS: PASSED.")
print("Pendientes controles complementarios y resumen final.")

metric,evaluated_checks,invalid_aggregate_amounts,different_checks,max_absolute_difference,status
Venta bruta,2142846,0,0,0.0,PASS
Venta registrada,2142846,0,0,0.0,PASS
Pagos,2142846,0,0,0.0,PASS
Propinas registradas,2142846,0,0,0.0,PASS
Voids,2142846,0,0,0.0,PASS
Descuentos de cuenta,2142846,0,0,0.0,PASS
Descuentos de artículo,2142846,0,0,0.0,PASS


BLOQUE DE IMPORTES DETALLE → CHECKS: PASSED.
Pendientes controles complementarios y resumen final.


In [0]:
# Calidad temporal de Silver checks.
# No convierte explícitamente a UTC ni modifica tablas.

temporal = (
    spark.table(full_table_name("silver_checks"))
    .withColumn(
        "_expected_business_date",
        F.expr("try_cast(BusinessDate AS DATE)"),
    )
    .withColumn(
        "_expected_open",
        F.expr("try_cast(OpenTime AS TIMESTAMP)"),
    )
    .withColumn(
        "_expected_close",
        F.expr("try_cast(CloseTime AS TIMESTAMP)"),
    )
)

temporal_rules = [
    (
        "Fecha de negocio original no convertible",
        "CRITICAL",
        F.col("_expected_business_date").isNull(),
    ),
    (
        "business_date no corresponde a BusinessDate",
        "CRITICAL",
        ~F.col("business_date").eqNullSafe(
            F.col("_expected_business_date")
        ),
    ),
    (
        "open_time_local no corresponde a OpenTime",
        "CRITICAL",
        ~F.col("open_time_local").eqNullSafe(F.col("_expected_open")),
    ),
    (
        "close_time_local no corresponde a CloseTime",
        "CRITICAL",
        ~F.col("close_time_local").eqNullSafe(F.col("_expected_close")),
    ),
    (
        "OpenTime ausente",
        "WARNING",
        F.col("OpenTime").isNull(),
    ),
    (
        "CloseTime ausente",
        "WARNING",
        F.col("CloseTime").isNull(),
    ),
    (
        "OpenTime presente pero no convertible",
        "WARNING",
        F.col("OpenTime").isNotNull()
        & F.col("_expected_open").isNull(),
    ),
    (
        "CloseTime presente pero no convertible",
        "WARNING",
        F.col("CloseTime").isNotNull()
        & F.col("_expected_close").isNull(),
    ),
    (
        "Cierre anterior a apertura",
        "WARNING",
        F.col("_expected_close") < F.col("_expected_open"),
    ),
    (
        "Apertura igual a cierre",
        "WARNING",
        F.col("_expected_open") == F.col("_expected_close"),
    ),
]

temporal_summary = temporal.agg(
    F.count("*").alias("total"),
    *[
        F.count(F.when(condition, 1)).alias(f"rule_{i}")
        for i, (_, _, condition) in enumerate(temporal_rules)
    ],
).first().asDict()

temporal_rows = []

for i, (rule, severity, _) in enumerate(temporal_rules):
    affected = int(temporal_summary[f"rule_{i}"])
    total = int(temporal_summary["total"])

    status = (
        "PASS" if affected == 0
        else "FAIL" if severity == "CRITICAL"
        else "WARN"
    )

    temporal_rows.append(
        (
            rule,
            severity,
            total,
            affected,
            round(100.0 * affected / total, 6) if total else None,
            status,
        )
    )

temporal_report = spark.createDataFrame(
    temporal_rows,
    schema=(
        "rule STRING, severity STRING, evaluated_rows LONG, "
        "affected_rows LONG, affected_pct DOUBLE, status STRING"
    ),
)

display(temporal_report)

failed_temporal_rules = [
    rule
    for rule, severity, total, affected, pct, status in temporal_rows
    if status == "FAIL"
]

if failed_temporal_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Consistencia temporal incumplida: "
        + "; ".join(failed_temporal_rules)
    )

print("BLOQUE TEMPORAL: SIN FALLOS CRÍTICOS.")
print("Revisar las advertencias del reporte, si existen.")

rule,severity,evaluated_rows,affected_rows,affected_pct,status
Fecha de negocio original no convertible,CRITICAL,2142846,0,0.0,PASS
business_date no corresponde a BusinessDate,CRITICAL,2142846,0,0.0,PASS
open_time_local no corresponde a OpenTime,CRITICAL,2142846,0,0.0,PASS
close_time_local no corresponde a CloseTime,CRITICAL,2142846,0,0.0,PASS
OpenTime ausente,WARNING,2142846,0,0.0,PASS
CloseTime ausente,WARNING,2142846,0,0.0,PASS
OpenTime presente pero no convertible,WARNING,2142846,0,0.0,PASS
CloseTime presente pero no convertible,WARNING,2142846,0,0.0,PASS
Cierre anterior a apertura,WARNING,2142846,0,0.0,PASS
Apertura igual a cierre,WARNING,2142846,52503,2.450153,WARN


BLOQUE TEMPORAL: SIN FALLOS CRÍTICOS.
Revisar las advertencias del reporte, si existen.


In [0]:
# Consistencia y frecuencia de las banderas de Silver checks.
# La presencia de una anomalía no bloquea.
# Una bandera calculada incorrectamente sí bloquea este control.

flag_checks = spark.table(full_table_name("silver_checks"))

flag_definitions = {
    "flag_no_items":
        F.col("item_line_count") == 0,

    "flag_no_payments":
        F.col("payment_count") == 0,

    "flag_empty_check":
        (F.col("item_line_count") == 0)
        & (F.col("payment_count") == 0),

    "flag_zero_duration":
        F.coalesce(
            F.col("open_time_local") == F.col("close_time_local"),
            F.lit(False),
        ),

    "flag_negative_duration":
        F.coalesce(
            F.col("close_time_local") < F.col("open_time_local"),
            F.lit(False),
        ),

    "flag_discount_both_levels":
        (F.col("check_discount_count") > 0)
        & (F.col("item_discount_count") > 0),
}

flag_expressions = [F.count("*").alias("total")]

for flag, expected in flag_definitions.items():
    flag_expressions.extend([
        F.count(
            F.when(F.col(flag) == True, 1)
        ).alias(f"{flag}_marked"),

        F.count(
            F.when(
                ~F.col(flag).eqNullSafe(expected)
                | F.col(flag).isNull(),
                1,
            )
        ).alias(f"{flag}_inconsistent"),
    ])

flag_summary = flag_checks.agg(
    *flag_expressions
).first().asDict()

flag_rows = []
total = int(flag_summary["total"])

for flag in flag_definitions:
    marked = int(flag_summary[f"{flag}_marked"])
    inconsistent = int(flag_summary[f"{flag}_inconsistent"])

    flag_rows.append((
        flag,
        total,
        marked,
        round(100.0 * marked / total, 6) if total else None,
        inconsistent,
        "PASS" if inconsistent == 0 else "FAIL",
    ))

flag_report = spark.createDataFrame(
    flag_rows,
    schema=(
        "flag STRING, evaluated_checks LONG, marked_checks LONG, "
        "marked_pct DOUBLE, inconsistent_checks LONG, status STRING"
    ),
)

display(flag_report)

failed_flag_rules = [
    row[0] for row in flag_rows if row[-1] == "FAIL"
]

if failed_flag_rules:
    raise RuntimeError(
        "QUALITY GATE: FAILED — Banderas inconsistentes: "
        + "; ".join(failed_flag_rules)
    )

print("BLOQUE DE CONSISTENCIA DE BANDERAS: PASSED.")
print("Los conteos de banderas son indicadores, no fraude confirmado.")

flag,evaluated_checks,marked_checks,marked_pct,inconsistent_checks,status
flag_no_items,2142846,11508,0.537043,0,PASS
flag_no_payments,2142846,3856,0.179948,0,PASS
flag_empty_check,2142846,1163,0.054274,0,PASS
flag_zero_duration,2142846,52503,2.450153,0,PASS
flag_negative_duration,2142846,0,0.0,0,PASS
flag_discount_both_levels,2142846,7629,0.356022,0,PASS


BLOQUE DE CONSISTENCIA DE BANDERAS: PASSED.
Los conteos de banderas son indicadores, no fraude confirmado.


In [0]:
# Resumen de los controles implementados.
# Ejecutar después de todas las celdas anteriores.

block_results = [
    (
        "Acceso a tablas",
        len(table_access_results),
        sum(row[1] == "FAIL" for row in table_access_results),
        0,
    ),
    (
        "Llaves de Silver checks",
        len(key_rule_results),
        sum(row[-1] == "FAIL" for row in key_rule_results),
        0,
    ),
    (
        "Reconciliación de checks Bronze → Silver",
        len(reconciliation_rows),
        sum(row[-1] == "FAIL" for row in reconciliation_rows),
        0,
    ),
    (
        "Integridad de eventos → checks",
        len(event_integrity_rows),
        sum(row[-1] == "FAIL" for row in event_integrity_rows),
        0,
    ),
    (
        "Conteos globales de eventos",
        len(event_count_rows),
        sum(row[-1] == "FAIL" for row in event_count_rows),
        0,
    ),
    (
        "Llaves de eventos con posición",
        len(event_position_rows),
        sum(row[-1] == "FAIL" for row in event_position_rows),
        0,
    ),
    (
        "Importes detalle → checks",
        len(amount_rows),
        sum(row[-1] == "FAIL" for row in amount_rows),
        0,
    ),
    (
        "Calidad temporal",
        len(temporal_rows),
        sum(row[-1] == "FAIL" for row in temporal_rows),
        sum(row[-1] == "WARN" for row in temporal_rows),
    ),
    (
        "Consistencia de banderas",
        len(flag_rows),
        sum(row[-1] == "FAIL" for row in flag_rows),
        0,
    ),
]

gate_summary_rows = [
    (
        block,
        int(rules),
        int(failures),
        int(warnings),
        "FAIL" if failures else "PASS_WITH_WARNINGS" if warnings else "PASS",
    )
    for block, rules, failures, warnings in block_results
]

gate_summary_report = spark.createDataFrame(
    gate_summary_rows,
    schema=(
        "block STRING, evaluated_rules LONG, failed_rules LONG, "
        "warning_rules LONG, status STRING"
    ),
)

display(gate_summary_report)

total_rules = sum(row[1] for row in block_results)
total_failures = sum(row[2] for row in block_results)
total_warnings = sum(row[3] for row in block_results)

print(f"Reglas evaluadas: {total_rules}")
print(f"Reglas fallidas: {total_failures}")
print(f"Reglas con advertencia: {total_warnings}")

print("\nALCANCE Y LIMITACIONES:")
print("- La aprobación aplica únicamente a los controles implementados.")
print("- Descuentos: sin validación de unicidad por posición.")
print("- Conteos de eventos Bronze/Silver: comparación global, no por check.")
print("- Importes: consistencia interna de Silver; no fidelidad contra Bronze.")
print("- Relaciones con dimensiones: pendientes de evaluar.")
print("- Inventario de tipos disponible; sin contrato automático de tipos.")
print("- Zona horaria del origen no verificada.")
print("- Estos controles no confirman ni descartan fraude.")

if total_failures:
    raise RuntimeError(
        f"QUALITY GATE: FAILED — {total_failures} reglas incumplidas."
    )

print("\nQUALITY GATE: PASSED")

block,evaluated_rules,failed_rules,warning_rules,status
Acceso a tablas,10,0,0,PASS
Llaves de Silver checks,9,0,0,PASS
Reconciliación de checks Bronze → Silver,3,0,0,PASS
Integridad de eventos → checks,16,0,0,PASS
Conteos globales de eventos,7,0,0,PASS
Llaves de eventos con posición,12,0,0,PASS
Importes detalle → checks,7,0,0,PASS
Calidad temporal,10,0,1,PASS_WITH_WARNINGS
Consistencia de banderas,6,0,0,PASS


Reglas evaluadas: 80
Reglas fallidas: 0
Reglas con advertencia: 1

ALCANCE Y LIMITACIONES:
- La aprobación aplica únicamente a los controles implementados.
- Descuentos: sin validación de unicidad por posición.
- Conteos de eventos Bronze/Silver: comparación global, no por check.
- Importes: consistencia interna de Silver; no fidelidad contra Bronze.
- Relaciones con dimensiones: pendientes de evaluar.
- Inventario de tipos disponible; sin contrato automático de tipos.
- Zona horaria del origen no verificada.
- Estos controles no confirman ni descartan fraude.

QUALITY GATE: PASSED
